In [ ]:
import torch
import time
import math
import triton
import triton.language as tl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HEAD_DIM = 64
VALUE_DIM = 64

SEQUENCE_LENGHTS = [4096,8192,16284,32768]
BLOCK_SIZES = [32,64]

WARMUP = 5
ITERATIONS = 20
torch.manual_seed(42)

print("="*15)
print("FLASH ATTENTION")
print("="*15)

print(f"\nDevice: {DEVICE}")
print(f"Head Dim: {HEAD_DIM}")
print(f"Value Dim: {VALUE_DIM}")


def standard_Attention(q,k,v):

    d = q.shape[-1]
    scores = q @ k.T / math.sqrt(d)
    weights = torch.softmax(scores,dim=-1)

    return weights @ v

def pytorch_sdpa(q,k,v):
    q = q[None,None,:,:]
    k = k[None,None,:,:]
    v = v[None,None,:,:]

    output = torch.nn.functional.scaled_dot_product_attention(q,k,v)

    return output[0,0]

def torch_online_softmax(q,k,v,block_size=64):

    n_q,d = q.shape
    n_k = k.shape[0]
    scale = 1.0/math.sqrt(d)

    running_max = q.new_full((n_q,1),-torch.inf)
    running_sum = q.new_zeros((n_q,1))

    accumulator = q.new_zeros((n_q,v.shape[1]))

    for start in range(0,n_k,block_size):
        k_tile = k[start:start+block_size]
        v_tile = v[start:start+block_size]

        scores = (q@k_tile.T)*scale
        new_max = torch.maximum(running_max,scores.max(dim=-1,keepdim=True).values)

        old_scale = torch.exp(running_max-new_max)

        weights = torch.exp(scores-new_max)

        running_sum = (
            old_scale * running_sum
            + weights.sum(dim=-1,keepdim=True)
        )

        accumulator = (
            old_scale * accumulator
            + weights @ v_tile
        )

        running_max = new_max

    return accumulator / running_sum



def flash_attention(q,k,v,kv_block_size=64):

    n_q,d = q.shape
    n_k = k.shape[0]

    scale = 1.0 / math.sqrt(d)

    running_max = q.new_full((n_q,1),-torch.inf)

    running_sum = q.new_zeros(n_q,1)

    accumulator = q.new_zeros(n_q,v.shape[1])

    for start in range(0,n_k,kv_block_size):
        k_tile = k[start:start+kv_block_size]
        v_tile = v[start:start+kv_block_size]

        scores = (q @ k_tile.T) * scale

        new_max = torch.maximum(
            running_max,
            scores.max(dim=-1,keepdim=True).values
        )

        old_scale = torch.exp(running_max - new_max)

        tile_weights = torch.exp(scores-new_max)

        running_sum = (
            old_scale * running_sum
            + tile_weights.sum(dim=-1,keepdim=True)
        )

        accumulator = (
            old_scale * accumulator
            + tile_weights @ v_tile
        )

        running_max = new_max

    return accumulator / running_sum

def generate_data(
        seq_len
):
    q = torch.randn(
        seq_len,
        64,
        device=DEVICE,
        dtype=torch.float16
    )

    k = torch.randn_like(q)
    v = torch.randn_like(q)

    return q.contiguous(), k.contiguous(), v.contiguous()

def test_correctness(q,k,v,block_size):

    with torch.inference_mode():
        output_standard = standard_Attention(q,k,v)

        output_flash = flash_attention(q,k,v,block_size)

    max_error = (
        output_standard-output_flash
    ).abs().max().item()

    mean_error = (
        output_standard-output_flash
    ).abs().mean().item()

    is_correct = torch.allclose(
        output_standard,
        output_flash,
        atol=1e-2,
        rtol=1e-2
    )

    return max_error,mean_error,is_correct

def benchmark(
        fn,
        warmup=WARMUP,
        iterations=ITERATIONS
):
    with torch.inference_mode():

        for _ in range(warmup):
            fn()

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        for _ in range(iterations):
            fn()

        if DEVICE=="cuda":
            torch.cuda.synchronize()

        end = time.perf_counter()

    avg_time = (end-start)/iterations

    return avg_time * 1000

def measure_peak_memory(fn):

    if DEVICE != "cuda":
        return None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    with torch.inference_mode():
        fn()
        torch.cuda.synchronize()

    peak_memory =  torch.cuda.max_memory_allocated()

    return peak_memory / (1024**2)

@triton.jit
def flash_attention_kernel(
    Q,K,V,O,
    N:tl.constexpr,
    D:tl.constexpr,
    DV:tl.constexpr,
    SCALE:tl.constexpr,
    BLOCK_M:tl.constexpr,
    BLOCK_N:tl.constexpr,
    BLOCK_D:tl.constexpr,
    BLOCK_DV:tl.constexpr
):

  pid = tl.program_id(0)

  offs_m = pid * BLOCK_M + tl.arange(0,BLOCK_M)

  offs_d = tl.arange(0,BLOCK_D)
  offs_dv = tl.arange(0,BLOCK_DV)

  offs_n = tl.arange(0,BLOCK_N)

  q = tl.load(
      Q + offs_m[:,None] * D + offs_d[None,:],
      mask = (
          (offs_m[:,None]<N)&
          (offs_d[None,:]<D)
      ),
      other = 0.0
  )

  m_i = tl.full((BLOCK_M,),float("-inf"),tl.float32)
  l_i = tl.zeros((BLOCK_M,),tl.float32)

  acc = tl.zeros((BLOCK_M,BLOCK_DV),tl.float32)

  for start_n in range(0, N, BLOCK_N):

    current_n = start_n + offs_n

    k = tl.load(
        K + current_n[None,:] * D + offs_d[:,None],
        mask = (
            (current_n[None,:]<N)&
            (offs_d[:,None]<D)
        ),
        other = 0.0
    )

    v = tl.load(
            V + current_n[:, None] * DV + offs_dv[None, :],
            mask=(
                (current_n[:, None] < N) &
                (offs_dv[None, :] < DV)
            ),
            other=0.0
        )

    qk = tl.dot(q,k,input_precision="ieee") * SCALE

    qk = tl.where(
        current_n[None,:]<N,
        qk,
        float("-inf")
    )

    m_ij = tl.maximum(
        m_i,
        tl.max(qk,axis=1)
    )

    alpha = tl.exp(m_i-m_ij)

    p = tl.exp(qk-m_ij[:,None])

    l_i = alpha * l_i + tl.sum(p,axis=1)

    acc = (acc*alpha[:,None]+tl.dot(p,v,input_precision="ieee"))

    m_i = m_ij

  output = acc / l_i[:,None]

  tl.store(
      O + offs_m[:,None] * DV + offs_dv[None,:],
      output,
      mask = (
          (offs_m[:,None]<N)&
          (offs_dv[None,:]<DV)
      )
  )

def triton_flash_attention(q,k,v,block_m=32,block_n = 32):

  n,d = q.shape
  dv = v.shape[1]

  assert q.is_cuda and k.is_cuda and v.is_cuda
  assert q.is_contiguous()
  assert k.is_contiguous()
  assert v.is_contiguous()
  assert q.shape[0] == k.shape[0]==v.shape[0]

  output = torch.empty(
      (n,dv),
      device = q.device,
      dtype = q.dtype
  )

  block_d = triton.next_power_of_2(d)
  block_dv = triton.next_power_of_2(dv)

  grid = (triton.cdiv(n,block_m),)

  flash_attention_kernel[grid](
      q,k,v,output,
      n,d,dv,
      1.0/math.sqrt(d),
      block_m,block_n,
      block_d,block_dv,
      num_warps=4
      )

  return output

def main():

    print("="*15)
    print("STARTING BENCHMARKS")
    print("="*15)

    for seq_len in SEQUENCE_LENGHTS:

        print(f"\n Sequence Length: {seq_len}")
        print("-"*10)

        q,k,v = generate_data(seq_len)

        max_error,mean_error,is_correct = test_correctness(q,k,v,BLOCK_SIZES[0])

        print("\nCORRECTNESS\n")
        print(f"Maximum Error: {max_error}")
        print(f"Mean Error: {mean_error}")
        print(f"Outputs Match: {is_correct}")

        if not is_correct:
            print("WARNING: Numerical mismatch detected")

        standard_time = benchmark(lambda:standard_Attention(q,k,v))
        standard_memory = measure_peak_memory(lambda:standard_Attention(q,k,v))

        print("\nStandard Attention")
        print(f"Execution Time: {standard_time:.4f} ms")

        if standard_memory is not None:
            print(f"Peak memory: {standard_memory:.4f} MB")

        print("\nFLASH ATTENTION")
        print(
            f"{'Block Size':<15}"
            f"{'Time(ms)':<15}"
            f"{'Speed-Up':<15}"
            f"{'Peak Memory':<15}"
        )

        for block_size in BLOCK_SIZES:
            flash_fn = lambda:triton_flash_attention(q,k,v,block_m=32,block_n=block_size)
            flash_time = benchmark(flash_fn)

            speedup = standard_time / flash_time

            flash_memory = measure_peak_memory(flash_fn)

            memory_str = (
                f"{flash_memory:.2f} MB"
                if flash_memory is not None
                else "N/A"
            )

            print(
                f"{block_size:<15}"
                f"{flash_time:<15}"
                f"{speedup:<15}"
                f"{memory_str:<15}"
            )

        print("-"*20)

if __name__ == "__main__":
    main()